# Tyson sloppy-subspace geometry

This notebook loads a saved Tyson NNSE neutral-set `.npz`, computes the local Hessian sloppy subspace at sampled neutral-set points, and compares those subspaces with principal angles. It is designed to test whether the sloppy directions are globally fixed or rotate along the neutral set.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "NNSE":
    ROOT = ROOT.parent
if not (ROOT / "wsbw_pipeline.py").exists():
    candidates = [Path.cwd() / "WhySystemsBiologyWorks", Path.cwd()]
    ROOT = next(path for path in candidates if (path / "wsbw_pipeline.py").exists())
sys.path.insert(0, str(ROOT))

from nnse_accessibility import choose_biggest_neutral_npz
from nnse_sloppy_subspace import SloppySubspaceConfig, analyze_tyson_sloppy_subspaces, save_sloppy_subspace

MODEL_KEY = "tyson1991"
# Leave None to select the largest available Tyson neutral-set npz.
NPZ_PATH = None
N_POINTS = 80          # Hessian computations; start with 30-80 locally, then increase if useful.
K_SLOPPY = 3
LOCAL_NEIGHBORS = 12
SEED = 42
T_END = 100.0
N_TIME = 501

if NPZ_PATH is None:
    NPZ_PATH = choose_biggest_neutral_npz(ROOT, MODEL_KEY)

print("ROOT:", ROOT)
print("NPZ_PATH:", NPZ_PATH)

The comparison uses three related quantities. Pairwise sloppy-subspace distance asks whether different neutral-set points have the same sloppy directions. Distance to the wild-type sloppy space asks whether the neutral set keeps following the original wild-type directions. Local tangent alignment asks whether the empirical neutral-set geometry, estimated by local PCA of nearby neutral points, aligns with the local Hessian sloppy subspace.

In [ ]:
config = SloppySubspaceConfig(
    model=MODEL_KEY,
    npz=str(NPZ_PATH),
    n_points=N_POINTS,
    k_sloppy=K_SLOPPY,
    local_neighbors=LOCAL_NEIGHBORS,
    seed=SEED,
    t_end=T_END,
    n_time=N_TIME,
)

result = analyze_tyson_sloppy_subspaces(config)
tag = f"{MODEL_KEY}_sloppy_subspace_{NPZ_PATH.parent.name}_n{N_POINTS}_k{K_SLOPPY}"
out_npz, out_json = save_sloppy_subspace(result, tag)

print("Saved:", out_npz)
print("Saved:", out_json)
print("Supported Hessian parameters:", list(result["supported_parameter_names"]))
print("Unsupported parameters ignored:", list(result["unsupported_parameter_names"]))
print("Valid Hessians:", result["summary"]["valid_hessians"], "/", result["summary"]["neutral_points_sampled"])
print("Median pairwise chordal distance:", result["summary"]["pairwise_chordal_median"])
print("Median distance to WT sloppy space:", result["summary"]["wt_chordal_median"])
print("Median local tangent/sloppy distance:", result["summary"]["tangent_chordal_median"])

In [ ]:
pairwise = result["pairwise_chordal"]
upper = pairwise[np.triu_indices_from(pairwise, k=1)]
upper = upper[np.isfinite(upper)]
wt = result["wt_chordal"]
tangent = result["tangent_chordal"]
logdist = result["log_distance_to_wt"]

fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)

ax = axes[0, 0]
ax.hist(upper, bins=30, color="#4C78A8", alpha=0.85)
ax.set_xlabel("pairwise sloppy-subspace distance")
ax.set_ylabel("count")
ax.set_title("Do sloppy spaces differ across neutral points?")

ax = axes[0, 1]
im = ax.imshow(pairwise, cmap="viridis", interpolation="nearest")
ax.set_title("Pairwise subspace-distance matrix")
ax.set_xlabel("sample index")
ax.set_ylabel("sample index")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[1, 0]
mask = np.isfinite(wt)
ax.scatter(logdist[mask], wt[mask], s=22, color="#F58518", alpha=0.8)
ax.set_xlabel("log-parameter distance to WT")
ax.set_ylabel("distance to WT sloppy space")
ax.set_title("Do sloppy directions rotate away from WT?")

ax = axes[1, 1]
vals = [wt[np.isfinite(wt)], tangent[np.isfinite(tangent)]]
ax.boxplot(vals, labels=["to WT sloppy", "to local tangent"], patch_artist=True,
           boxprops=dict(facecolor="#72B7B2", alpha=0.55), medianprops=dict(color="black"))
ax.set_ylabel("subspace distance")
ax.set_title("Reference and local-tangent alignment")

plt.show()

In [ ]:
eigvals = result["eigvals"]
valid = result["valid"]
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
for row in eigvals[valid]:
    ax.semilogy(np.arange(1, len(row) + 1), np.maximum(row, 1e-30), color="black", alpha=0.18)
ax.semilogy(np.arange(1, eigvals.shape[1] + 1), np.nanmedian(np.maximum(eigvals[valid], 1e-30), axis=0),
            color="#E45756", lw=2.0, label="median")
ax.set_xlabel("Hessian eigenvalue rank, stiff to sloppy")
ax.set_ylabel("eigenvalue")
ax.set_title("Local Tyson Hessian spectra across sampled neutral points")
ax.legend(frameon=False)
plt.show()

Interpretation: large pairwise sloppy-subspace distances imply that the local sloppy directions rotate across the neutral set. Low local tangent/sloppy distances imply that the neutral set locally follows the local sloppy space. Together these would support the stronger "curved thread" story: the neutral set is extended along sloppy directions, and those directions vary through parameter space.